In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [3]:
import sys, os

sys.path.append(os.environ['LP_ROOT'])
import project
from project.core.utils import pprint

sys.path.append(os.environ['PROJECT'] + '/param_search')
import param_search as ps

ps.set_verbose(False)
ps.set_backend('sge')
ps.api.get_queue()

# Gather examples

In [4]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
for p in data_root.iterdir():
    print(p)

/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Processed
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-22.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/subject_files.txt
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/Images
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/sample1000_2025-07-17.csv
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/DISK_USAGE
/restricted/projectnb/batmanlab/mragoza/data/COPDGene/ClinicalData


In [5]:
import pandas as pd
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
subject_list

['16514P',
 '20748Q',
 '11007Z',
 '14771Z',
 '13651K',
 '15900P',
 '15312Y',
 '21042H',
 '21877G',
 '10127E',
 '23577E',
 '13816Q',
 '21559S',
 '25335Q',
 '23023N',
 '10572Z',
 '10887Y',
 '21410K',
 '17920F',
 '14684E',
 '16132B',
 '17862R',
 '11746L',
 '25728J',
 '20609C',
 '16637F',
 '13297S',
 '25767T',
 '25695U',
 '21572K',
 '15078Q',
 '15297C',
 '14857J',
 '13460D',
 '18840M',
 '12506W',
 '11498S',
 '21611U',
 '15088T',
 '12831H',
 '19612E',
 '19784H',
 '10217F',
 '23037Y',
 '25130Y',
 '15623P',
 '19027T',
 '14380K',
 '10212V',
 '16060C',
 '19356M',
 '22357L',
 '25923H',
 '21189L',
 '24608U',
 '15204V',
 '11743F',
 '11879E',
 '12477P',
 '10815Z',
 '17790S',
 '15489L',
 '11850G',
 '13034M',
 '17255W',
 '14632L',
 '26066U',
 '18184E',
 '15532M',
 '19558Y',
 '20640W',
 '21741H',
 '18015H',
 '17137Q',
 '12422Q',
 '15699W',
 '21933Q',
 '18397V',
 '14550J',
 '20703U',
 '21004Z',
 '24581A',
 '24331D',
 '17505T',
 '15894U',
 '16977D',
 '18935X',
 '19410S',
 '18515B',
 '21399W',
 '21899Q',

In [6]:
import project.datasets.copdgene
dataset = project.datasets.copdgene.COPDGeneDataset(data_root)
examples = dataset.list_examples(subject_list, state_pairs=[('EXP', 'INSP')])
len(examples)

1000

In [64]:
base_dir = '2026-08-08_preprocess'

template = '''\
#!/bin/bash -l
#SBATCH --job-name={job_name}
#SBATCH --account=asc170022p
#SBATCH --partition=GPU-shared
#SBATCH --gres=gpu:1
#SBATCH -t 6:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects=[{subject}] \\
    --set dataset.examples.variant={variant} \\

'''
name_format = '{params_hash}'

grid = ps.param_grid(
    config='2026-08-07_config.yaml',
    data_name='COPDGene',
    data_root=str(data_root),
    subject=[ex.subject for ex in examples][:10],
    variant='2026-08-08'
)
len(grid)

10

In [65]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=True)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,log_dir,stdout_path,stderr_path,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.variant
0,7f789eee4046a70b,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",7f789eee4046a70b,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,2026-08-08
1,d002ad4e92447715,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",d002ad4e92447715,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,2026-08-08
2,159f0be13cc26027,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",159f0be13cc26027,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,2026-08-08
3,540aeaf79732fafd,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",540aeaf79732fafd,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,2026-08-08
4,bf2428c889f5f9ed,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",bf2428c889f5f9ed,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,2026-08-08
5,10167ceacb989277,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",10167ceacb989277,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,15900P,2026-08-08
6,4019f381249985f8,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",4019f381249985f8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,15312Y,2026-08-08
7,a599da5e2e0c234f,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",a599da5e2e0c234f,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,21042H,2026-08-08
8,c340f44edd852bd4,NEW,0,<NA>,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,<NA>,<NA>,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",c340f44edd852bd4,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,21877G,2026-08-08
9,f9e04773db2100c6,NEW,0,<NA>,<NA>,<NA>,<N

In [55]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,7f789eee4046a70b,PENDING,1,7085317,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,16514P,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
1,d002ad4e92447715,PENDING,1,7085318,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,20748Q,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
2,159f0be13cc26027,PENDING,1,7085319,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,11007Z,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
3,540aeaf79732fafd,PENDING,1,7085320,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,14771Z,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
4,bf2428c889f5f9ed,PENDING,1,7085321,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,13651K,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
5,10167ceacb989277,PENDING,1,7085322,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,15900P,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
6,4019f381249985f8,PENDING,1,7085323,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,15312Y,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
7,a599da5e2e0c234f,PENDING,1,7085324,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,21042H,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
8,c340f44edd852bd4,PENDING,1,7085325,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,21877G,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>
9,f9e04773db2100c6,PENDING,1,7085326,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,10127E,2026-08-08,NaN,2026-08-08T01:06:43,status,NaN,NaN,False,<NA>,<NA>


In [56]:
print(jobs['stderr'].iloc[0])

<NA>


In [43]:
jobs.loc[:, 'job_id'] = pd.NA

In [63]:
jobs = ps.submit(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,log_dir,stdout_path,stderr_path,params_json,params_hash,params.config,params.data_name,params.data_root,params.subject,params.variant
0,7f789eee4046a70b,SUBMITTED,1,7085460,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",7f789eee4046a70b,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,16514P,2026-08-08
1,d002ad4e92447715,SUBMITTED,1,7085461,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",d002ad4e92447715,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,20748Q,2026-08-08
2,159f0be13cc26027,SUBMITTED,1,7085462,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",159f0be13cc26027,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,11007Z,2026-08-08
3,540aeaf79732fafd,SUBMITTED,1,7085463,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",540aeaf79732fafd,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,14771Z,2026-08-08
4,bf2428c889f5f9ed,SUBMITTED,1,7085464,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",bf2428c889f5f9ed,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,13651K,2026-08-08
5,10167ceacb989277,SUBMITTED,1,7085465,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",10167ceacb989277,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,15900P,2026-08-08
6,4019f381249985f8,SUBMITTED,1,7085466,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,"{""config"": ""2026-08-07_config.yaml"", ""data_nam...",4019f381249985f8,/projectnb/batmanlab/mragoza/lung-project/note...,COPDGene,/restricted/projectnb/batmanlab/mragoza/data/C...,15312Y,2026-08-08
7,a599da5e2e0c234f,SUBMITTED,1,7085467,<NA>,<NA>,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,/projectnb/batma